# Celeb-DF v2 cross-dataset test: standalone branches vs. gated fusion

Purpose: find out **which view generalises** from FaceForensics++ C23 to Celeb-DF v2, and tighten the confidence interval by using the **full official test list** (518 videos) instead of a 200-video sample.

What this notebook does
- Evaluates the already-trained models only: the gated model, the **separately trained spatial and frequency ResNet18 models**, late fusion, and the gated model with alpha forced to 1.0 / 0.5 / 0.0.
- **No training, no fine-tuning, no threshold tuning.** The threshold stays fixed at 0.5. Do not pick a threshold from these results.
- Face crops are cached on Drive, so reruns do not repeat face detection.

Run on a Colab GPU (Runtime > Change runtime type > T4 GPU). Run the cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install mediapipe kagglehub scikit-learn
!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O /content/blaze_face_short_range.tflite
print("Setup complete.")

In [ ]:
import os, shutil, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

P = "/content/drive/MyDrive/Multimedia_Authentication"
MODELS_DIR = f"{P}/v2/models"
CKPTS = {
    "gated":     f"{MODELS_DIR}/gated_v2_best.pth",
    "spatial":   f"{MODELS_DIR}/spatial_resnet18_v2_best.pth",
    "frequency": f"{MODELS_DIR}/frequency_resnet18_v2_best.pth",
}
OUT = f"{P}/v2/results"
FIG = f"{OUT}/figs"
FACE_CACHE = f"{P}/cross_dataset_cache/faces_celebdf"
for d in (OUT, FIG, FACE_CACHE):
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
for name, path in CKPTS.items():
    print(f"{name:10s} checkpoint exists:", os.path.exists(path))
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing checkpoint: {path}. Update CKPTS to the exact file.")
if device.type != "cuda":
    warnings.warn("GPU not detected. This will be slow on CPU.")

## 1. Face extraction (identical to the training and earlier cross-dataset pipeline)
8 evenly spaced frames, largest detected face, nearest box reused when a frame has no detection, 1.4x square crop resized to 224x224.

In [ ]:
def make_detector():
    options = vision.FaceDetectorOptions(
        base_options=mp_python.BaseOptions(model_asset_path="/content/blaze_face_short_range.tflite"),
        min_detection_confidence=0.5
    )
    return vision.FaceDetector.create_from_options(options)


def read_frames(video_path, n=8):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    frames = []
    for frame_idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def detect_box(detector, frame, max_side=640):
    height, width = frame.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    small = (cv2.resize(frame, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
             if scale < 1.0 else frame)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                      data=np.ascontiguousarray(small, dtype=np.uint8)))
    if not result.detections:
        return None
    box = max(result.detections, key=lambda d: d.bounding_box.width * d.bounding_box.height).bounding_box
    return (box.origin_x / scale, box.origin_y / scale, box.width / scale, box.height / scale)


def square_crop(frame, box, scale=1.4, out=224):
    height, width = frame.shape[:2]
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    side = int(min(max(w, h) * scale, height, width))
    side = max(side, 2)
    x1 = int(np.clip(cx - side / 2, 0, width - side))
    y1 = int(np.clip(cy - side / 2, 0, height - side))
    crop = frame[y1:y1 + side, x1:x1 + side]
    if crop.size == 0:
        raise ValueError("Empty face crop")
    return cv2.resize(crop, (out, out), interpolation=cv2.INTER_AREA)


def process_video(detector, video_path):
    frames = read_frames(video_path)
    if frames is None:
        return None, "read_failed"
    boxes = [detect_box(detector, frame) for frame in frames]
    found = [i for i, b in enumerate(boxes) if b is not None]
    if not found:
        return None, "no_face"
    for i, b in enumerate(boxes):
        if b is None:
            boxes[i] = boxes[min(found, key=lambda j: abs(j - i))]
    crops = [square_crop(f, b) for f, b in zip(frames, boxes)]
    return np.stack(crops).astype(np.uint8), "ok"

## 2. Load the trained models (strict loading: a key mismatch fails loudly)

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def haar_highfreq(x):
    a = x[..., 0::2, 0::2]; b = x[..., 0::2, 1::2]
    c = x[..., 1::2, 0::2]; d = x[..., 1::2, 1::2]
    lh = (a + b - c - d) / 2
    hl = (a - b + c - d) / 2
    hh = (a - b - c + d) / 2
    return torch.stack([lh.mean(1), hl.mean(1), hh.mean(1)], dim=1)


def to_branch(x, branch):
    # x: [B, T, 3, H, W] in 0..1
    B, T, C, H, W = x.shape
    x = x.reshape(B * T, C, H, W)
    if branch == "spatial":
        x = (x - MEAN) / STD
    else:
        h = haar_highfreq(x)
        mn = h.amin(dim=(1, 2, 3), keepdim=True)
        mx = h.amax(dim=(1, 2, 3), keepdim=True)
        h = (h - mn) / (mx - mn + 1e-8)
        h = F.interpolate(h, size=(H, W), mode="bilinear", align_corners=False)
        x = (h - MEAN) / STD
    return x.reshape(B, T, C, H, W)


def backbone():
    return nn.Sequential(*list(resnet18(weights=None).children())[:-1])


class BranchNet(nn.Module):
    # Standalone single-branch model (same layout as training: backbone.*, classifier.*)
    def __init__(self):
        super().__init__()
        self.backbone = backbone()
        self.classifier = nn.Linear(512, 2)

    def forward(self, x):
        B, T = x.shape[:2]
        f = self.backbone(x.flatten(0, 1)).flatten(1)
        return self.classifier(f.view(B, T, 512).mean(1))


class GatedFusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial_bb = backbone()
        self.freq_bb = backbone()
        self.gate = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))

    def feats(self, x, bb):
        B, T = x.shape[:2]
        return bb(x.flatten(0, 1)).flatten(1).view(B, T, 512).mean(1)


def load(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    model.to(device).eval()
    print(f"loaded {Path(path).name} | val AUC in checkpoint: {ck.get('val_auc')}")
    return model


gated = load(GatedFusionNet(), CKPTS["gated"])
spatial_net = load(BranchNet(), CKPTS["spatial"])
freq_net = load(BranchNet(), CKPTS["frequency"])


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


@torch.no_grad()
def predict_all(frames):
    x = (torch.from_numpy(frames).to(device).unsqueeze(0)
         .permute(0, 1, 4, 2, 3).float().div(255.0))
    with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
        xs, xf = to_branch(x, "spatial"), to_branch(x, "frequency")

        # gated model: compute branch features once, then mix
        s = gated.feats(xs, gated.spatial_bb)
        f = gated.feats(xf, gated.freq_bb)
        alpha = torch.sigmoid(gated.gate(torch.cat([s, f], 1)).float())

        def mix(a):
            a = a.to(s.dtype)
            return gated.classifier(a * s + (1 - a) * f)

        lg_gated = mix(alpha)
        lg_a1 = mix(torch.full_like(alpha, 1.0))
        lg_a05 = mix(torch.full_like(alpha, 0.5))
        lg_a0 = mix(torch.full_like(alpha, 0.0))

        # separately trained standalone branches
        lg_s = spatial_net(xs)
        lg_f = freq_net(xf)

    pf = lambda lg: float(torch.softmax(lg.float(), 1)[0, 1])
    p_s, p_f = pf(lg_s), pf(lg_f)
    p_late = float(1 / (1 + np.exp(-(logit(p_s) + logit(p_f)) / 2)))
    return {
        "p_gated": pf(lg_gated), "alpha": float(alpha[0, 0]),
        "p_spatial": p_s, "p_frequency": p_f, "p_late": p_late,
        "p_alpha1": pf(lg_a1), "p_alpha05": pf(lg_a05), "p_alpha0": pf(lg_a0),
    }

## 3. Official Celeb-DF v2 test list
`N_PER_CLASS = None` uses **all 518 official test videos** (178 real, 340 fake). Set it to a number (e.g. 100) for a quick balanced sample. Videos already downloaded to Drive are reused.

In [ ]:
import kagglehub

KAGGLE_DATASET = "reubensuju/celeb-df-v2"
N_PER_CLASS = None          # None = full official test list; or e.g. 100 for a balanced sample
RANDOM_SEED = 42
CDF_LOCAL = f"{P}/cross_dataset_cache/Celeb-DF-v2"
os.makedirs(CDF_LOCAL, exist_ok=True)


def locate_downloaded_file(download_result, expected_name):
    result_path = Path(download_result)
    if result_path.is_file():
        return str(result_path)
    if result_path.is_dir():
        direct = result_path / expected_name
        if direct.is_file():
            return str(direct)
        matches = list(result_path.rglob(Path(expected_name).name))
        if len(matches) == 1:
            return str(matches[0])
    raise FileNotFoundError(f"Could not locate {expected_name} inside: {download_result}")


def kaggle_download_one(remote_candidates, expected_name):
    last_error = None
    for remote_path in remote_candidates:
        try:
            result = kagglehub.dataset_download(KAGGLE_DATASET, path=remote_path)
            return locate_downloaded_file(result, expected_name)
        except Exception as exc:
            last_error = exc
            print(f"Could not fetch {remote_path}: {type(exc).__name__}")
    raise RuntimeError(f"Unable to download {expected_name}. Last error: {last_error}")


list_local = os.path.join(CDF_LOCAL, "List_of_testing_videos.txt")
if not os.path.isfile(list_local):
    shutil.copy2(kaggle_download_one(
        ["celeb-df-v2/List_of_testing_videos.txt", "List_of_testing_videos.txt"],
        "List_of_testing_videos.txt"), list_local)


def parse_official_test_list(list_path):
    records = []
    with open(list_path, "r", encoding="utf-8") as fh:
        for n, raw in enumerate(fh, start=1):
            line = raw.strip()
            if not line:
                continue
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(f"Unexpected format at line {n}: {line}")
            flag, rel = parts
            rel = rel.strip().lstrip("/")
            # Official protocol: 1 = REAL, 0 = FAKE
            if flag == "1":
                label = "REAL"
            elif flag == "0":
                label = "FAKE"
            else:
                raise ValueError(f"Unknown label flag '{flag}' at line {n}")
            low = rel.lower()
            if "synthesis" in low and label != "FAKE":
                raise ValueError(f"Label/path mismatch at line {n}: {line}")
            if ("celeb-real" in low or "youtube-real" in low) and label != "REAL":
                raise ValueError(f"Label/path mismatch at line {n}: {line}")
            records.append({"relative_path": rel, "label": label})
    out = pd.DataFrame(records).drop_duplicates("relative_path")
    if out.empty:
        raise ValueError("Empty test list.")
    return out


manifest = parse_official_test_list(list_local)
print("Official test-list videos:", len(manifest))
print(manifest["label"].value_counts())

rng_sel = np.random.default_rng(RANDOM_SEED)
parts = []
for label in ["REAL", "FAKE"]:
    g = manifest[manifest["label"] == label]
    take = len(g) if N_PER_CLASS is None else min(N_PER_CLASS, len(g))
    idx = rng_sel.choice(g.index.to_numpy(), size=take, replace=False)
    parts.append(g.loc[idx])
selected = pd.concat(parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print("\nSelected:", len(selected))
print(selected["label"].value_counts())

## 4. Download the selected MP4 files (cached on Drive; only missing files are fetched)

In [ ]:
def safe_relative_path(rel):
    norm = os.path.normpath(rel)
    if norm.startswith("..") or os.path.isabs(norm):
        raise ValueError(f"Unsafe relative path: {rel}")
    return norm


items, download_errors = [], []
for row in tqdm(selected.itertuples(index=False), total=len(selected), desc="Preparing videos"):
    rel = safe_relative_path(row.relative_path)
    local_path = os.path.join(CDF_LOCAL, rel)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    if not os.path.isfile(local_path) or os.path.getsize(local_path) == 0:
        if rel.startswith("celeb-df-v2/"):
            candidates = [rel, rel[len("celeb-df-v2/"):]]
        else:
            candidates = [f"celeb-df-v2/{rel}", rel]
        try:
            src = kaggle_download_one(candidates, os.path.basename(rel))
            shutil.copy2(src, local_path)
        except Exception as exc:
            download_errors.append({"relative_path": rel, "label": row.label, "error": repr(exc)})
            continue

    if os.path.isfile(local_path) and os.path.getsize(local_path) > 0:
        items.append((local_path, rel, row.label))

print(f"\nReady videos: {len(items)} / {len(selected)} | download failures: {len(download_errors)}")
if download_errors:
    pd.DataFrame(download_errors).to_csv(f"{OUT}/celebdf_download_errors.csv", index=False)
if not items:
    raise RuntimeError("No videos downloaded. Check KaggleHub access.")
print(pd.Series([l for _, _, l in items]).value_counts())

## 5. Run inference (all models, one pass per video)

In [ ]:
detector = make_detector()
rows = []

for local_path, rel, label in tqdm(items, desc="Evaluating"):
    cache_file = os.path.join(FACE_CACHE, rel.replace("/", "__").replace("\\", "__") + ".npy")
    try:
        if os.path.isfile(cache_file):
            faces, status = np.load(cache_file), "ok"
        else:
            faces, status = process_video(detector, local_path)
            if faces is not None:
                np.save(cache_file, faces)
        if faces is None:
            rows.append({"path": rel, "label": label, "status": status})
            continue
        res = predict_all(faces)
        rows.append({"path": rel, "label": label, "status": "ok", **res})
    except Exception as exc:
        rows.append({"path": rel, "label": label, "status": f"error: {type(exc).__name__}: {exc}"})

pred = pd.DataFrame(rows)
csv_path = f"{OUT}/cross_CelebDFv2_standalone_branches_predictions.csv"
pred.to_csv(csv_path, index=False)

valid = pred[pred["status"] == "ok"].copy()
print("Saved:", csv_path)
print(f"Evaluated {len(valid)} / {len(pred)} videos | skipped {len(pred) - len(valid)}")
print(pred.loc[pred.status != "ok", ["path", "label", "status"]].to_string(index=False))
print(valid["label"].value_counts())

## 6. Results at the fixed threshold 0.5
Rows marked "forced alpha" use the gated model's classifier with the mixing weight fixed by hand, so they show how much the learned gate adds. AUC is threshold-free. Do not choose a threshold from this table.

In [ ]:
y = (valid["label"].values == "FAKE").astype(int)
real_idx, fake_idx = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
rng = np.random.default_rng(42)

MODELS = {
    "Gated fusion (learned alpha)":  "p_gated",
    "Spatial branch alone":          "p_spatial",
    "Frequency branch alone":        "p_frequency",
    "Late fusion (mean of logits)":  "p_late",
    "Gated, alpha forced 1.0":       "p_alpha1",
    "Gated, alpha forced 0.5":       "p_alpha05",
    "Gated, alpha forced 0.0":       "p_alpha0",
}
B = 1000
boot_idx = [np.concatenate([rng.choice(real_idx, len(real_idx), replace=True),
                            rng.choice(fake_idx, len(fake_idx), replace=True)]) for _ in range(B)]


def boot_ci(p):
    aucs = [roc_auc_score(y[i], p[i]) for i in boot_idx]
    return np.percentile(aucs, [2.5, 97.5])


rows = []
for name, col in MODELS.items():
    p = valid[col].to_numpy()
    pred_lab = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred_lab, labels=[0, 1]).ravel()
    real_recall, fake_recall = tn / (tn + fp), tp / (tp + fn)
    lo, hi = boot_ci(p)
    rows.append({
        "model": name,
        "AUC": round(roc_auc_score(y, p), 4),
        "AUC 95% CI": f"{lo:.3f}-{hi:.3f}",
        "accuracy": round((pred_lab == y).mean(), 4),
        "REAL recall": round(real_recall, 4),
        "FAKE recall": round(fake_recall, 4),
        "balanced acc": round((real_recall + fake_recall) / 2, 4),
        "pred FAKE %": round(100 * pred_lab.mean(), 1),
    })

results = pd.DataFrame(rows)
print(f"Celeb-DF v2 official test list | REAL {len(real_idx)} | FAKE {len(fake_idx)} | threshold 0.5\n")
print(results.to_string(index=False))
results.to_csv(f"{OUT}/cross_CelebDFv2_standalone_branches_results.csv", index=False)

# Paired bootstrap: is a model really different from the gated model in AUC?
print("\nPaired AUC differences vs. gated model (95% CI; 'YES' only if the interval excludes 0):")
pg = valid["p_gated"].to_numpy()
for name, col in MODELS.items():
    if col == "p_gated":
        continue
    pm = valid[col].to_numpy()
    d = [roc_auc_score(y[i], pm[i]) - roc_auc_score(y[i], pg[i]) for i in boot_idx]
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{name:32s} mean {np.mean(d):+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  significant: {'YES' if (lo > 0 or hi < 0) else 'no'}")

print("\nMean alpha  REAL %.3f | FAKE %.3f" % (
    valid.loc[y == 0, "alpha"].mean(), valid.loc[y == 1, "alpha"].mean()))

## 7. Figures for the report

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name in ["Gated fusion (learned alpha)", "Spatial branch alone", "Frequency branch alone", "Late fusion (mean of logits)"]:
    fpr, tpr, _ = roc_curve(y, valid[MODELS[name]].to_numpy())
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y, valid[MODELS[name]]):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Celeb-DF v2 (trained on FF++ C23)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig(f"{FIG}/celebdf_roc_standalone.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (title, col) in zip(axes, [("Gated: P(fake)", "p_gated"), ("Spatial branch: P(fake)", "p_spatial"),
                                   ("Frequency branch: P(fake)", "p_frequency")]):
    ax.hist(valid.loc[y == 0, col], bins=25, alpha=0.6, label="REAL", range=(0, 1))
    ax.hist(valid.loc[y == 1, col], bins=25, alpha=0.6, label="FAKE", range=(0, 1))
    ax.set_title(title); ax.set_xlabel("probability"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG}/celebdf_prob_distributions.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.hist(valid.loc[y == 0, "alpha"], bins=25, alpha=0.6, label="REAL")
ax.hist(valid.loc[y == 1, "alpha"], bins=25, alpha=0.6, label="FAKE")
ax.set_xlabel("alpha (spatial weight)"); ax.set_title("Gate weight on Celeb-DF v2"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG}/celebdf_alpha.png", dpi=200); plt.show()
print("Figures saved to:", FIG)

## 8. Save a summary

In [ ]:
summary = {
    "dataset": "Celeb-DF v2",
    "split": "official List_of_testing_videos.txt",
    "n_per_class_setting": N_PER_CLASS,
    "videos_evaluated": int(len(valid)),
    "real": int((y == 0).sum()), "fake": int((y == 1).sum()),
    "evaluation_only": True, "fine_tuning": False, "threshold": 0.5,
    "checkpoints": CKPTS,
}
with open(f"{OUT}/cross_CelebDFv2_standalone_branches_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))

## How to report this honestly
- State the numbers as measured, at threshold 0.5, on the official test list.
- Treat reasons for the drop (higher-quality fakes, a different face-swap pipeline, different compression, generator-specific frequency cues) as **hypotheses**, not proven causes.
- Do not select a threshold on this test set. A threshold chosen here would have to be labelled as tuned on Celeb-DF.
- The forced-alpha rows describe the same gated model with a hand-fixed mix; they are not separately trained models.